In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize, lsq_linear
from scipy.special import expit
from sklearn.model_selection import KFold
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']
TAG = 'se_hurdle_s'
DEST = OUTPUT/TAG
DEST.mkdir(exist_ok=True)
RHO_GRID = np.linspace(0, .98, 51)
N_SPLITS = 5

In [3]:
# original SE fit cohort; no missing years converted to zeros
panel = pd.read_csv(INPUT/'panel_complete.csv', index_col='dblp_id')
Q_EMP = panel.to_numpy(dtype=float)
N_EMP = len(panel)
pd.DataFrame([{'model':TAG, 'fit_cohort':'complete careers', 'scholars':N_EMP,
               'transitions':N_EMP*Y, 'initialization':'empirical complete-cohort q0',
               'source_commit':'3b3e9e36fcb06380cbd545c413db609d9f7eb3ba'}]).to_csv(DEST/'fit_support.csv', index=False)

In [4]:
STAGE_ORDER = ['0', '1-4', '5-7', '8-20']
STAGE_TRANSITIONS = {
    '0': np.array([0]),
    '1-4': np.arange(1, 5),
    '5-7': np.arange(5, 8),
    '8-20': np.arange(8, 20)}

def transition_stage(t):
    if t == 0:
        return '0'
    if 1 <= t <= 4:
        return '1-4'
    if 5 <= t <= 7:
        return '5-7'
    if 8 <= t <= 19:
        return '8-20'
    raise ValueError(f'No stage for transition year {t}')

def target_stage(target_year):
    return transition_stage(target_year - 1)

def history_panel(q, rho):
    n, years = q.shape
    history = np.zeros_like(q, dtype=float)
    numerator = np.zeros(n)
    denominator = np.zeros(n)

    for target_year in range(1, years):
        history[:, target_year] = np.divide(numerator,denominator,out=np.zeros_like(numerator),where=denominator > 0)
        current_year = target_year - 1
        numerator = rho * numerator + np.log1p(q[:, current_year])
        denominator = rho * denominator + 1.0

    return history

def build_transition_rows(q, history):
    rows = []

    for target_year in range(1, q.shape[1]):
        transition_year = target_year - 1
        q_prev = q[:, transition_year]
        q_now = q[:, target_year]

        rows.append(pd.DataFrame({
            'scholar': np.arange(q.shape[0]),
            'transition_year': transition_year,'target_year': target_year,
            'stage': transition_stage(transition_year),
            'q_prev': q_prev,'q': q_now,
            'active': (q_now > 0).astype(int),
            'x_prev': np.log1p(q_prev),
            'prev_active': (q_prev > 0).astype(int),
            'restart': (q_prev <= 0).astype(int),
            'history': history[:, target_year]}))

    return pd.concat(rows, ignore_index=True)

In [5]:
CONTINUOUS = {'x_prev', 'history'}
RIDGE = 1e-6
MIN_PROB = 1e-9

def fit_scaler(data, feature_names):
    means = {}
    scales = {}

    for name in feature_names:
        if name in CONTINUOUS:
            means[name] = float(data[name].mean())
            scale = float(data[name].std(ddof=0))
            scales[name] = scale if np.isfinite(scale) and scale > 1e-8 else 1.0
        else:
            means[name] = 0.0
            scales[name] = 1.0

    return means, scales

def design_matrix(data, feature_names, means, scales):
    columns = [np.ones(len(data))]

    for name in feature_names:
        columns.append((data[name].to_numpy(dtype=float) - means[name]) / scales[name])

    return np.column_stack(columns)

def array_design(values, spec):
    n = len(next(iter(values.values())))
    columns = [np.ones(n)]

    for name in spec['feature_names']:
        columns.append((np.asarray(values[name], dtype=float) - spec['means'][name])/ spec['scales'][name])

    return np.column_stack(columns)

def fit_logistic(X, y, constrained_index=None):
    def objective(weights):
        eta = np.clip(X @ weights, -35, 35)
        p = expit(eta)
        nll = -np.sum(y * np.log(p + 1e-12)+ (1 - y) * np.log(1 - p + 1e-12))
        nll += RIDGE * np.sum(weights[1:] ** 2)

        gradient = X.T @ (p - y)
        gradient[1:] += 2 * RIDGE * weights[1:]
        return nll, gradient

    bounds = [(None, None)] * X.shape[1]
    if constrained_index is not None:
        bounds[constrained_index] = (0, None)

    result = minimize(objective,np.zeros(X.shape[1]),jac=True,method='L-BFGS-B',bounds=bounds)

    if not result.success:
        raise RuntimeError(result.message)

    return result.x

def fit_stage(stage_data, use_history=True, constrain_history=True):
    activity_features = ['x_prev', 'prev_active']
    positive_features = ['x_prev', 'restart']

    if use_history:
        activity_features.append('history')
        positive_features.append('history')

    act_means, act_scales = fit_scaler(stage_data, activity_features)
    X_act = design_matrix(stage_data, activity_features, act_means, act_scales)
    y_act = stage_data['active'].to_numpy(dtype=float)

    act_history_index = None
    if use_history and constrain_history:
        act_history_index = 1 + activity_features.index('history')

    activity_coef = fit_logistic(X_act,y_act,constrained_index=act_history_index)

    positive = stage_data[stage_data['active'].eq(1)].copy()
    pos_means, pos_scales = fit_scaler(positive, positive_features)
    X_pos = design_matrix(positive, positive_features, pos_means, pos_scales)
    y_pos = np.log(positive['q'].to_numpy(dtype=float))

    lower = np.full(X_pos.shape[1], -np.inf)
    upper = np.full(X_pos.shape[1], np.inf)

    if use_history and constrain_history:
        lower[1 + positive_features.index('history')] = 0.0

    positive_fit = lsq_linear(X_pos,y_pos,bounds=(lower, upper),lsq_solver='exact')

    positive_coef = positive_fit.x
    residuals = y_pos - X_pos @ positive_coef
    residuals = residuals - residuals.mean()
    sigma = max(float(np.sqrt(np.mean(residuals ** 2))), 1e-8)
    laplace_scale = max(float(np.mean(np.abs(residuals))), 1e-8)

    return {'activity': {
            'feature_names': activity_features,
            'means': act_means,
            'scales': act_scales,
            'coef': activity_coef,
            'n': len(stage_data)},
        'positive': {
            'feature_names': positive_features,
            'means': pos_means,
            'scales': pos_scales,
            'coef': positive_coef,
            'residuals': residuals,
            'sigma': sigma,
            'laplace_scale': laplace_scale,
            'n': len(positive)}}

def fit_model(q, history, use_history=True, constrain_history=True):
    rows = build_transition_rows(q, history)

    return {stage: fit_stage(rows[rows['stage'].eq(stage)],use_history=use_history,constrain_history=constrain_history) for stage in STAGE_ORDER}

def score_model(model, q, history):
    rows = build_transition_rows(q, history)
    nll = 0.0
    n = 0

    for stage in STAGE_ORDER:
        stage_data = rows[rows['stage'].eq(stage)]
        fitted = model[stage]

        X_act = design_matrix(stage_data,fitted['activity']['feature_names'],fitted['activity']['means'],fitted['activity']['scales'])
        p = np.clip(expit(X_act @ fitted['activity']['coef']),MIN_PROB,1 - MIN_PROB)
        y = stage_data['active'].to_numpy(dtype=float)
        nll -= np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        n += len(stage_data)

        positive = stage_data[stage_data['active'].eq(1)]
        X_pos = design_matrix(positive,fitted['positive']['feature_names'],fitted['positive']['means'],fitted['positive']['scales'])
        y_pos = np.log(positive['q'].to_numpy(dtype=float))
        sigma = fitted['positive']['sigma']
        standardized = (y_pos - X_pos @ fitted['positive']['coef']) / sigma

        nll += np.sum(0.5 * standardized ** 2+ np.log(sigma) + 0.5 * np.log(2 * np.pi))

    return nll / n

def original_scale_coefficients(spec):
    names = spec['feature_names']
    standardized = spec['coef']
    original = {'intercept': float(standardized[0])}

    for j, name in enumerate(names, start=1):
        original[name] = float(standardized[j] / spec['scales'][name])
        original['intercept'] -= (standardized[j]* spec['means'][name]/ spec['scales'][name])

    return original

In [6]:
# scholar folds select rho; selection score is not an independent test score
folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(Q_EMP))
profile_rows, fold_rows = [], []
for rho in RHO_GRID:
    history = history_panel(Q_EMP, rho)
    scores = []
    for fold, (train_idx, test_idx) in enumerate(folds):
        fitted = fit_model(Q_EMP[train_idx], history[train_idx])
        score = score_model(fitted, Q_EMP[test_idx], history[test_idx])
        scores.append(score)
        fold_rows.append({'rho':rho, 'fold':fold, 'mean_logspace_nll':score})
    profile_rows.append({'rho':rho, 'mean_logspace_nll':np.mean(scores),
                         'se_nll':np.std(scores, ddof=1)/np.sqrt(N_SPLITS)})
profile = pd.DataFrame(profile_rows)
RHO_HAT = float(profile.loc[profile.mean_logspace_nll.idxmin(), 'rho'])
profile.to_csv(DEST/'rho_profile.csv', index=False)
pd.DataFrame(fold_rows).to_csv(DEST/'rho_fold_scores.csv', index=False)
display(profile.nsmallest(5, 'mean_logspace_nll'))

,rho,mean_logspace_nll,se_nll
34,0.6664,1.087330,0.008516
35,0.6860,1.087342,0.008526
33,0.6468,1.087403,0.008506
36,0.7056,1.087448,0.008537
32,0.6272,1.087554,0.008497


In [7]:
history = history_panel(Q_EMP, RHO_HAT)
model = fit_model(Q_EMP, history)
rows = []
for stage in STAGE_ORDER:
    for equation in ['activity', 'positive']:
        spec = model[stage][equation]
        rows.append({'stage':stage, 'equation':equation, 'n':spec['n'],
                     **original_scale_coefficients(spec), 'sigma':spec.get('sigma', np.nan), 'rho':RHO_HAT})
params = pd.DataFrame(rows)
params.to_csv(DEST/'parameters.csv', index=False)
pd.DataFrame([{'rho':RHO_HAT, 'half_life':np.log(.5)/np.log(RHO_HAT) if RHO_HAT>0 else 0,
               'seed':SEED, 'n_splits':N_SPLITS}]).to_csv(DEST/'memory.csv', index=False)
display(params)

,stage,equation,n,intercept,x_prev,prev_active,history,sigma,rho,restart
0,0,activity,510,0.485508,1.111501,-0.780557,0.000000,NaN,0.6664,NaN
1,0,positive,397,0.860750,0.520505,NaN,0.000000,0.577512,0.6664,0.765671
2,1-4,activity,2040,-0.014620,0.939176,-0.504497,0.635710,NaN,0.6664,NaN
3,1-4,positive,1741,0.812442,0.381944,NaN,0.236117,0.613305,0.6664,0.552137
4,5-7,activity,1530,-0.480062,0.962003,-0.992276,1.282260,NaN,0.6664,NaN
5,5-7,positive,1357,0.379710,0.359612,NaN,0.399294,0.596172,0.6664,0.571413
6,8-20,activity,6120,-0.893912,1.191152,-0.879455,1.372544,NaN,0.6664,NaN
7,8-20,positive,5365,0.017668,0.434154,NaN,0.486292,0.587889,0.6664,0.523566
